In [1]:
print("ram ram")

ram ram


In [2]:
# !pip install kafka-python

In [5]:
import json

from datetime import datetime

from kafka import KafkaConsumer

import time



# 1. Helper function to decode Debezium's 19-digit nanosecond timestamps

def parse_debezium_time(nano_ts):

    if not nano_ts:

        return None

    try:

        # Convert nanoseconds to seconds for Python's datetime

        dt = datetime.fromtimestamp(nano_ts / 1e9)

        return dt.strftime('%Y-%m-%d %H:%M:%S')

    except Exception:

        return str(nano_ts)



# 2. Dynamic Group ID for testing (forces a fresh read every time you run the cell)

dynamic_group_id = f'mai-agent-listener23-{int(time.time())}'



# Initialize the Kafka Consumer

consumer = KafkaConsumer(

    'mes_server.mes_new.dbo.WorkOrder',

    bootstrap_servers=['localhost:9092'],

    auto_offset_reset='earliest',

    enable_auto_commit=True,

    group_id=dynamic_group_id,

    value_deserializer=lambda x: json.loads(x.decode('utf-8')) if x else None

)



print("🎧 Listening for WorkOrder changes... (Optimized for AI Agents)")



try:

    for message in consumer:

        payload = message.value.get('payload') if message.value else None

       

        # Skip tombstone (null) records

        if not payload:

            continue

           

        op = payload.get('op')

        before = payload.get('before') or {}

        after = payload.get('after') or {}

       

        wo_number = after.get('WorkOrderNumber') if after else before.get('WorkOrderNumber')

       

        # 3. Create a structured memory payload for your LLM/Agent

        ai_event_payload = {

            "event_type": None,

            "work_order": wo_number,

            "changes": {}

        }



        print("-" * 60)

       

        if op == 'r':

            print(f"📘 SNAPSHOT: WorkOrder {wo_number} cached.")

           

        elif op == 'c':

            ai_event_payload["event_type"] = "CREATED"

            print(f"🟢 INSERT: New WorkOrder {wo_number} created.")

           

        elif op == 'u':

            ai_event_payload["event_type"] = "UPDATED"

            print(f"🟡 UPDATE: WorkOrder {wo_number} modified.")

           

            for key, old_val in before.items():

                new_val = after.get(key)

               

                if old_val != new_val and key not in ['UpdatedDate', 'CreatedDate']:

                   

                    # Intercept and format Time/Date fields

                    if any(time_keyword in key for time_keyword in ['Date', 'Start', 'End']):

                        old_val = parse_debezium_time(old_val)

                        new_val = parse_debezium_time(new_val)



                    print(f"   ➔ [{key}] changed from '{old_val}' to '{new_val}'")

                   

                    # Log the exact delta for the AI Agent

                    ai_event_payload["changes"][key] = {"from": old_val, "to": new_val}

                   

        elif op == 'd':

            ai_event_payload["event_type"] = "DELETED"

            print(f"🔴 DELETE: WorkOrder {wo_number} removed.")



        # --- AI AGENT INTEGRATION POINT ---

        # If this is a live update with actual changes, it is ready to be routed!

        if op == 'u' and ai_event_payload["changes"]:

            # Example:

            # mai_monitoring_agent.invoke(ai_event_payload)

            # websocket_server.broadcast(json.dumps(ai_event_payload))

            pass



except KeyboardInterrupt:

    print("\n🛑 Stopped listening to Kafka stream.")

C:\Users\renuk\AppData\Local\Temp\ipykernel_14952\2254883195.py:41: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


🎧 Listening for WorkOrder changes... (Optimized for AI Agents)
------------------------------------------------------------
📘 SNAPSHOT: WorkOrder WO202608030001 cached.
------------------------------------------------------------
📘 SNAPSHOT: WorkOrder WO202608030002 cached.
------------------------------------------------------------
🟡 UPDATE: WorkOrder WO202608030001 modified.
   ➔ [PlannedStart] changed from '2026-08-03 05:30:00' to '2026-08-03 00:00:00'
   ➔ [PlannedEnd] changed from '2026-08-11 05:30:00' to '2026-08-12 00:00:00'
   ➔ [Notes] changed from 'None' to 'sfzxcdzxfds'
------------------------------------------------------------
🟡 UPDATE: WorkOrder WO202608030001 modified.
   ➔ [PlannedStart] changed from '2026-08-03 00:00:00' to '2026-08-02 18:30:00'
   ➔ [PlannedEnd] changed from '2026-08-12 00:00:00' to '2026-08-20 18:30:00'
   ➔ [Notes] changed from 'None' to 'sfzxcdzxfds'
------------------------------------------------------------
🟡 UPDATE: WorkOrder WO202608030001 m

In [6]:
import json
import requests
from datetime import datetime
from kafka import KafkaConsumer
import time

# Helper function to decode Debezium's 19-digit nanosecond timestamps
def parse_debezium_time(nano_ts):
    if not nano_ts:
        return None
    try:
        dt = datetime.fromtimestamp(nano_ts / 1e9)
        return dt.strftime('%Y-%m-%d %H:%M:%S')
    except Exception:
        return str(nano_ts)

# Generate a fresh group ID
dynamic_group_id = f'mai-live-listener-{int(time.time())}'

# Initialize the Kafka Consumer
consumer = KafkaConsumer(
    'mes_server.mes_new.dbo.WorkOrder',
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='latest', # <--- THE FIX: Ignores all historical data
    enable_auto_commit=True,
    group_id=dynamic_group_id, 
    value_deserializer=lambda x: json.loads(x.decode('utf-8')) if x else None
)

# Webhook configuration
WEBHOOK_URL = "http://localhost:8000/api/webhook/workorder" # Change to your actual backend URL

print("🎧 Listening for LIVE WorkOrder changes only... (Old logs ignored)")

try:
    for message in consumer:
        payload = message.value.get('payload') if message.value else None
        
        if not payload:
            continue
            
        op = payload.get('op')
        
        # Explicitly skip historical snapshots just to be safe
        if op == 'r':
            continue
            
        before = payload.get('before') or {}
        after = payload.get('after') or {}
        wo_number = after.get('WorkOrderNumber') if after else before.get('WorkOrderNumber')
        
        ai_event_payload = {
            "event_type": None,
            "work_order": wo_number,
            "timestamp": datetime.now().isoformat(),
            "changes": {}
        }

        print("-" * 60)
        
        if op == 'c':
            ai_event_payload["event_type"] = "CREATED"
            print(f"🟢 LIVE INSERT: New WorkOrder {wo_number} created.")
            
        elif op == 'u':
            ai_event_payload["event_type"] = "UPDATED"
            print(f"🟡 LIVE UPDATE: WorkOrder {wo_number} modified.")
            
            for key, old_val in before.items():
                new_val = after.get(key)
                
                if old_val != new_val and key not in ['UpdatedDate', 'CreatedDate']:
                    if any(time_keyword in key for time_keyword in ['Date', 'Start', 'End']):
                        old_val = parse_debezium_time(old_val)
                        new_val = parse_debezium_time(new_val)

                    print(f"   ➔ [{key}] changed from '{old_val}' to '{new_val}'")
                    ai_event_payload["changes"][key] = {"from": old_val, "to": new_val}
                    
        elif op == 'd':
            ai_event_payload["event_type"] = "DELETED"
            print(f"🔴 LIVE DELETE: WorkOrder {wo_number} removed.")

        # --- WEBHOOK DISPATCHER ---
        # Only send the webhook if there was a meaningful update, insert, or delete
        if op in ['c', 'd'] or (op == 'u' and ai_event_payload["changes"]):
            try:
                # Fire and forget the webhook to your backend
                response = requests.post(WEBHOOK_URL, json=ai_event_payload, timeout=3)
                print(f"   📡 Webhook sent! Status Code: {response.status_code}")
            except requests.exceptions.RequestException as e:
                print(f"   ⚠️ Webhook failed to send: {e}")

except KeyboardInterrupt:
    print("\n🛑 Stopped listening to Kafka stream.")

C:\Users\renuk\AppData\Local\Temp\ipykernel_14952\932495508.py:21: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


🎧 Listening for LIVE WorkOrder changes only... (Old logs ignored)
------------------------------------------------------------
🟡 LIVE UPDATE: WorkOrder WO202608030001 modified.
   ➔ [PlannedStart] changed from '2026-06-30 04:30:00' to '2026-06-29 23:00:00'
   ➔ [PlannedEnd] changed from '2026-08-11 04:30:00' to '2026-08-19 23:00:00'
   ➔ [Notes] changed from 'None' to 'sfzxcdzxfds'
   ⚠️ Webhook failed to send: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/webhook/workorder (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
------------------------------------------------------------
🟡 LIVE UPDATE: WorkOrder WO202608100002 modified.
   ➔ [DueDate] changed from '2026-08-05 13:00:00' to '2026-08-05 07:30:00'
   ➔ [MachineId] changed from 'None' to '95'
   ➔ [PlannedStart] changed from '2026-08-

In [ ]:
#     🎧 Listening for LIVE WorkOrder changes only... (Old logs ignored)
# ------------------------------------------------------------
# 🟡 LIVE UPDATE: WorkOrder WO202608030001 modified.
#    ➔ [PlannedStart] changed from '2026-06-30 10:00:00' to '2026-06-30 04:30:00'
#    ➔ [PlannedEnd] changed from '2026-08-27 10:00:00' to '2026-08-11 04:30:00'
#    ➔ [Notes] changed from 'None' to 'sfzxcdzxfds'
#    ⚠️ Webhook failed to send: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/webhook/workorder (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
# ------------------------------------------------------------
# 🟢 LIVE INSERT: New WorkOrder WO202608100002 created.
#    ⚠️ Webhook failed to send: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/webhook/workorder (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
# ------------------------------------------------------------
# 🟡 LIVE UPDATE: WorkOrder WO202608100002 modified.
#    ➔ [PlannedQty] changed from 'AId/gA==' to 'FLIwq4A='
#    ➔ [DueDate] changed from '2026-08-06 00:00:00' to '2026-08-05 18:30:00'
#    ➔ [PlannedStart] changed from '2026-08-27 16:37:00' to '2026-08-27 11:07:00'
#    ➔ [PlannedEnd] changed from '2026-08-28 16:37:00' to '2026-08-28 11:07:00'
#    ➔ [UpdatedBy] changed from 'None' to 'shivi.gupta1614@gmail.com'
#    ⚠️ Webhook failed to send: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/webhook/workorder (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))
# ------------------------------------------------------------
# 🟡 LIVE UPDATE: WorkOrder WO202608100002 modified.
#    ➔ [DueDate] changed from '2026-08-05 18:30:00' to '2026-08-05 13:00:00'
#    ➔ [PriorityId] changed from '2' to '3'
#    ➔ [PlannedStart] changed from '2026-08-27 11:07:00' to '2026-08-27 05:37:00'
#    ➔ [PlannedEnd] changed from '2026-08-28 11:07:00' to '2026-08-28 05:37:00'
#    ⚠️ Webhook failed to send: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /api/webhook/workorder (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

# 🛑 Stopped listening to Kafka stream.